## Setting up search

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

In [2]:
df_ground_truth.head(5)

,question,document
0,I just found this course late — am I still all...,74eb249bbf
1,Is it too late to start the course if I missed...,74eb249bbf
2,Can I enroll after the course has already star...,74eb249bbf
3,"If I join the course now, can I still get a ce...",74eb249bbf
4,What’s the deadline for getting the certificat...,74eb249bbf


In [4]:
ground_truth[3]

{'question': 'If I join the course now, can I still get a certificate?',
 'document': '74eb249bbf'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
        
documents = documents_llm
index = build_index(documents)

In [10]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

## Collecting relevance data

In [8]:
q = ground_truth[0]
q

{'question': 'I just found this course late — am I still allowed to join in now?',
 'document': '74eb249bbf'}

In [12]:
doc_id = q['document']
results = text_search(query=q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I st

In [13]:
## Compare retrieved doc IDs with correct doc ID
for d in results:
    print(f'{d['id']} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
85384a18e5 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
c2903069a0 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [14]:
relevance = []

for d in results:
    relevance.append(int(d['id'] == doc_id))
    
relevance

[1, 0, 0, 0, 0]

In [ ]:
## Wrap it in the function
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [16]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

I just found this course late — am I still allowed to join in now?


[1, 0, 0, 0, 0]

In [17]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)

Where do I follow the LLM Zoomcamp syllabus, homework, and deadlines in one place?


[1, 0, 0, 0, 0]

In [19]:
from tqdm.auto import tqdm

## Do the same thing for all ground truth questions
def compute_relevance_total_text(ground_truth):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)
        
    return relevance_total